### Installations

In [5]:
%pip install snowflake-connector-python
%pip install ingest
%pip install nbimporter

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement ingest (from versions: none)
ERROR: No matching distribution found for ingest


Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import time
import datetime
import logging
from logging import getLogger
from snowflake.ingest import SimpleIngestManager, StagedFile
from requests import HTTPError
from cryptography.hazmat.primitives.serialization import load_pem_private_key, Encoding, PrivateFormat, NoEncryption
from cryptography.hazmat.backends import default_backend

# Setup logging
log_dir = r"c:\tmp"
os.makedirs(log_dir, exist_ok=True)
logging.basicConfig(filename=os.path.join(log_dir, "ingest.log"), level=logging.DEBUG)
logger = getLogger(__name__)

# Set passphrase for private key
os.environ["private_key_phrase"] = 'Phanipykey@66'

def get_private_key_passphrase():
    return os.getenv("private_key_phrase")

# Load encrypted private key
private_key_path = r"C:\Users\2320859\OneDrive - Cognizant\Desktop\Spotify_ETL_PIPELINE\Data_Loading\rsa_key_encrypted.p8"
if not os.path.exists(private_key_path):
    raise FileNotFoundError(f"The file '{private_key_path}' does not exist.")

with open(private_key_path, 'rb') as pem_in:
    pemlines = pem_in.read()
    private_key_obj = load_pem_private_key(
        pemlines,
        password=get_private_key_passphrase().encode(),
        backend=default_backend()
    )

private_key_text = private_key_obj.private_bytes(
    Encoding.PEM,
    PrivateFormat.PKCS8,
    NoEncryption()
).decode('utf-8')

# Snowflake connection details
account = 'HENVDEC-QO50665'
host = 'HENVDEC-QO50665.snowflakecomputing.com'
user = 'Phaninarina'

# Define pipe and file mappings
pipe_file_map = {
    'SPOTIFY_BRONZE_DB.SPOTIFY_BRONZE_SCHEMA.USERS_BRONZE_PIPE': 'users_data.csv',
    'SPOTIFY_BRONZE_DB.SPOTIFY_BRONZE_SCHEMA.SUBSCRIPTIONS_BRONZE_PIPE': 'subscriptions_data.csv',
    'SPOTIFY_BRONZE_DB.SPOTIFY_BRONZE_SCHEMA.ARTISTS_BRONZE_PIPE': 'artists_data.csv',
    'SPOTIFY_BRONZE_DB.SPOTIFY_BRONZE_SCHEMA.ALBUMS_BRONZE_PIPE': 'albums_data.csv',
    'SPOTIFY_BRONZE_DB.SPOTIFY_BRONZE_SCHEMA.SONGS_BRONZE_PIPE': 'songs_data.csv',
    'SPOTIFY_BRONZE_DB.SPOTIFY_BRONZE_SCHEMA.STREAM_ACTIVITY_BRONZE_PIPE': 'stream_activity_data.csv'
}

# Ingest each file using its corresponding pipe
for pipe_name, file_name in pipe_file_map.items():
    print(f"\nStarting ingestion for pipe: {pipe_name} with file: {file_name}")
    ingest_manager = SimpleIngestManager(
        account=account,
        host=host,
        user=user,
        pipe=pipe_name,
        private_key=private_key_text
    )

    staged_file_list = [StagedFile(file_name, None)]

    try:
        resp = ingest_manager.ingest_files(staged_file_list)
        assert resp['responseCode'] == 'SUCCESS'
        logger.info(f"Ingestion initiated for {file_name} via {pipe_name}")
        print(f"Ingestion initiated successfully for {file_name}")
    except HTTPError as e:
        logger.error(f"HTTP Error during ingestion for {file_name}: {e}")
        print(f"HTTP Error during ingestion for {file_name}: {e}")
        continue

    # Poll for ingestion history
    print("Polling for ingestion history...")
    while True:
        history_resp = ingest_manager.get_history()
        if history_resp['files']:
            print('Ingest Report:\n', history_resp)
            break
        else:
            time.sleep(20)

    # Optional: Get ingestion history for the past hour
    hour = datetime.timedelta(hours=1)
    date = datetime.datetime.utcnow() - hour
    history_range_resp = ingest_manager.get_history_range(date.isoformat() + 'Z')
    print('\nHistory scan report:\n', history_range_resp)



Starting ingestion for pipe: SPOTIFY_BRONZE_DB.SPOTIFY_BRONZE_SCHEMA.USERS_BRONZE_PIPE with file: users_data.csv
